In [1]:
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import (
    AutoConfig, 
    AutoModelForCausalLM, 
    AutoTokenizer,
    get_cosine_schedule_with_warmup
)

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading xLSTM 7B...")
xlstm_config = AutoConfig.from_pretrained("NX-AI/xLSTM-7b")
xlstm_config.step_kernel = "native"
xlstm_config.chunkwise_kernel = "chunkwise--native_autograd"
xlstm_config.sequence_kernel = "native_sequence__native"


xlstm_config.hidden_size = 768
xlstm_config.embedding_dim = 768
xlstm_config.num_heads = 4
xlstm_config.num_hidden_layers = 12
xlstm_config.num_blocks = 12

Loading xLSTM 7B...


In [3]:
with torch.device("cuda"):
    xlstm = AutoModelForCausalLM.from_config(config=xlstm_config)

tokenizer = AutoTokenizer.from_pretrained("NX-AI/xLSTM-7b")
tokenizer.pad_token = tokenizer.eos_token

In [4]:
total_params = xlstm.num_parameters()
print(f"Total parameters: {total_params:,}")

Total parameters: 162,303,840


In [14]:
# dataset = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")

# def clean_text(text_list):
#     return "\n".join([t.strip() for t in text_list if t.strip() != ""])

# def clean_and_tokenize(batch):
#     cleaned_texts = [clean_text(text) for text in batch["text"]]
#     return tokenizer(cleaned_texts)

# tokenized_dataset = dataset.map(
#     clean_and_tokenize,
#     batched=True,         
#     batch_size=10000,      
#     remove_columns=["text"]
# )

# train_tokens = tokenized_dataset["train"]["input_ids"]
# valid_tokens = tokenized_dataset["validation"]["input_ids"]
# test_tokens  = tokenized_dataset["test"]["input_ids"]

# tokenized_dataset.save_to_disk("./wikitext-103-tokenized")

In [ ]:
# from datasets import load_from_disk
# import itertools
# tokenized_dataset = load_from_disk("./wikitext-103-tokenized")

# def flatten_tokens(split):
#     token_iterator = (row for row in tokenized_dataset[split]["input_ids"])
#     flat_list = list(itertools.chain.from_iterable(token_iterator))
#     continuous_tensor = torch.tensor(flat_list, dtype=torch.int32)
#     print(f"Total tokens in continuous stream: {len(continuous_tensor):,}")
#     del flat_list
#     torch.save(continuous_tensor, f"{split}_tokens_list.pt")

# flatten_tokens("train")
# flatten_tokens("validation")
# flatten_tokens("test")


In [5]:
import torch
from torch.utils.data import DataLoader

train_tensor = torch.load("train_tokens_list.pt")
print(f"Loaded {len(train_tensor):,} train tokens.")
val_tensor = torch.load("validation_tokens_list.pt")
test_tensor = torch.load("test_tokens_list.pt")

Loaded 867,971,053 train tokens.


In [6]:
from torch.utils.data import Dataset, DataLoader
import random

class RandomSampleDataset(Dataset):
    def __init__(self, continuous_tokens, total_steps, window_size=512):
            self.tokens = continuous_tokens
            self.total_steps = total_steps
            self.window_size = window_size
            
            self.max_start_idx = len(self.tokens) - self.window_size

    def __len__(self):
        return self.total_steps

    def __getitem__(self, idx):
        # 1. Pick a random starting point
        start = random.randint(0, self.max_start_idx)
        
        # 2. Slice the exact window size (e.g., 512)
        chunk = self.tokens[start : start + self.window_size].to(torch.long)
        
        # 3. Return a dictionary matching Hugging Face kwargs
        return {
            "input_ids": chunk,
            # .clone() is critical here to prevent PyTorch from tripping over 
            # shared memory references during the internal loss backward pass.
            "labels": chunk.clone() 
        }

In [11]:
import math
import torch

@torch.no_grad() 
def evaluate_perplexity(model, test_tensor, context_length=128, batch_size=8):
    model.eval()
    
    total_loss = 0.0
    total_batches = 0
    
    # Calculate how many full context windows fit in the test tensor
    num_windows = len(test_tensor) // context_length
    
    # Truncate the end of the tensor so it divides perfectly by context_length
    truncated_test = test_tensor[:num_windows * context_length]
    
    # Reshape the 1D tensor into a 2D tensor of shape [num_windows, context_length]
    # This automatically splits the continuous stream into perfect sequential chunks
    eval_chunks = truncated_test.view(num_windows, context_length).to(torch.long)
    
    # Process the chunks in batches
    for i in range(0, num_windows, batch_size):
        batch_chunks = eval_chunks[i : i + batch_size].to(model.device)
        
        # Hugging Face models expect identical input_ids and labels for causal LM loss
        inputs = {"input_ids": batch_chunks, "labels": batch_chunks}
        
        outputs = model(**inputs)
        total_loss += outputs.loss.item()
        total_batches += 1
        
    avg_loss = total_loss / total_batches
    perplexity = math.exp(avg_loss)
    
    model.train() # Immediately put the model back into training mode!
    return perplexity




In [12]:
# Target: Effective Batch Size of 256

CONTEXT_LENGTH = 128
TRAINING_STEPS = 100
WARMUP_STEPS = 10
MAX_LR = 1e-3
WEIGHT_DECAY = 0.1

EFFECTIVE_BATCH_SIZE = 256
MICRO_BATCH_SIZE = 64
ACCUMULATION_STEPS = EFFECTIVE_BATCH_SIZE // MICRO_BATCH_SIZE

TOTAL_FORWARD_PASSES = TRAINING_STEPS * ACCUMULATION_STEPS


dataset = RandomSampleDataset(
    continuous_tokens=train_tensor, 
    total_steps=TOTAL_FORWARD_PASSES, 
    window_size=CONTEXT_LENGTH
)

dataloader = DataLoader(dataset, batch_size=MICRO_BATCH_SIZE, shuffle=False)

In [13]:
import os
optimizer = torch.optim.AdamW(xlstm.parameters(), lr=MAX_LR)
scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TRAINING_STEPS
)

xlstm.train()
optimizer.zero_grad() 

os.makedirs("model_checkpoints", exist_ok=True)

In [14]:
# ... (Dataset, Dataloader, Optimizer setup from previous steps)

xlstm.train()
optimizer.zero_grad()
global_step = 0 

print("\n--- Step 0 (Baseline) Evaluation ---")
baseline_ppl = evaluate_perplexity(
    model=xlstm, 
    test_tensor=test_tensor, 
    context_length=CONTEXT_LENGTH, 
    batch_size=64
)
print(f"Untrained Validation Perplexity: {baseline_ppl:.2f}")

for step, batch in enumerate(dataloader):
    batch = {k: v.to(xlstm.device) for k, v in batch.items()}
    
    outputs = xlstm(**batch)
    loss = outputs.loss / ACCUMULATION_STEPS
    loss.backward()
    
    # When we complete a full effective batch (Accumulation step)
    if (step + 1) % ACCUMULATION_STEPS == 0:
        torch.nn.utils.clip_grad_norm_(xlstm.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        
        global_step += 1
        
        # -------------------------------------------------------------
        # THE 1000-STEP EVALUATION & CHECKPOINT BLOCK
        # -------------------------------------------------------------
        if global_step % 100 == 0:
            print(f"\n--- Step {global_step} Evaluation ---")
            
            # 1. Calculate Perplexity
            ppl = evaluate_perplexity(
                model=xlstm, 
                test_tensor=test_tensor, 
                context_length=CONTEXT_LENGTH, 
                batch_size=64
            )
            print(f"Validation Perplexity: {ppl:.2f}")
            
            # 2. Save Model Weights
            checkpoint_dir = f"xlstm_checkpoints/step_{global_step}"
            print(f"Saving checkpoint to {checkpoint_dir}...")
            
            # Because you are using a Hugging Face model variation (xLSTM), 
            # save_pretrained is the expert standard. It saves the config.json 
            # and safetensors, making it trivially easy to load later via .from_pretrained()
            xlstm.save_pretrained(checkpoint_dir)
            
            print("--------------------------------\n")


--- Step 0 (Baseline) Evaluation ---


KeyboardInterrupt: 